# Phase 3 — Model Training

## Goal
Fine-tune a pretrained BART model on the XSum dataset to learn how to generate summaries.

## Approach: Transfer Learning + Fine-Tuning
- BART is already pretrained on millions of texts → understands language
- We fine-tune it on our dataset → specializes in news summarization
- Much more efficient than training from scratch

## Why BART?
| Model | Suitable? | Reason |
|---|---|---|
| RNN/LSTM | Weak | Struggles with long texts |
| BERT | Partial | Encoder only, not generative |
| GPT | Partial | Decoder only, not optimized for summarization |
| BART | Excellent | Encoder + Decoder, designed for summarization |

In [6]:
import os
os.environ["HF_HOME"] = r"D:\hf_cache"

from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    BartForConditionalGeneration,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    DataCollatorForSeq2Seq
)
import torch

## 1. Load Tokenized Dataset

In [7]:
train_data = load_from_disk("../data/processed/train_tokenized")
test_data  = load_from_disk("../data/processed/test_tokenized")

print(f"Train: {len(train_data)} examples")
print(f"Test:  {len(test_data)} examples")

Train: 1600 examples
Test:  400 examples


## 2. Load BART Model

`BartForConditionalGeneration` → BART version for text generation tasks.
"Conditional" means: generate text **conditioned on** an input (given article → generate summary).

In [8]:
MODEL_NAME = "facebook/bart-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model     = BartForConditionalGeneration.from_pretrained(MODEL_NAME)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
model.to(device)

Loading weights: 100%|██████████| 259/259 [00:00<00:00, 3907.38it/s]


Using device: cpu


BartForConditionalGeneration(
  (model): BartModel(
    (shared): BartScaledWordEmbedding(50265, 768, padding_idx=1)
    (encoder): BartEncoder(
      (embed_tokens): BartScaledWordEmbedding(50265, 768, padding_idx=1)
      (embed_positions): BartLearnedPositionalEmbedding(1026, 768)
      (layers): ModuleList(
        (0-5): 6 x BartEncoderLayer(
          (self_attn): BartAttention(
            (k_proj): Linear(in_features=768, out_features=768, bias=True)
            (v_proj): Linear(in_features=768, out_features=768, bias=True)
            (q_proj): Linear(in_features=768, out_features=768, bias=True)
            (out_proj): Linear(in_features=768, out_features=768, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=768, out_features=3072, bias=True)
          (fc2): Linear(in_features=3072, out_features=768, bias=True)
          (final_layer_n

## 3. Data Collator

`DataCollatorForSeq2Seq` applies **dynamic padding** — pads each batch only to the longest example in that batch.
Better than padding everything to max_length → saves memory and speeds up training.

In [9]:
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    padding=True
)

## 4. Training Arguments (Hyperparameters)

| Parameter | Value | Reason |
|---|---|---|
| num_train_epochs | 3 | Enough for fine-tuning, more may cause overfitting |
| per_device_train_batch_size | 2 | Small because BART is large |
| warmup_steps | 50 | Gradually increase LR to prevent harmful early updates |
| weight_decay | 0.01 | Regularization to prevent overfitting |
| fp16 | False | CPU does not support 16-bit precision |

In [10]:
training_args = Seq2SeqTrainingArguments(
    output_dir="../outputs/model",

    num_train_epochs=3,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    warmup_steps=50,
    weight_decay=0.01,

    logging_dir="../outputs/plots",
    logging_steps=10,

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,

    predict_with_generate=True,
    fp16=False,
)

[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


## 5. Trainer

`Seq2SeqTrainer` handles the full training loop automatically:
1. **Forward pass** → input goes through model → output generated
2. **Loss calculation** → how wrong was the prediction?
3. **Backward pass** → compute gradients
4. **Weight update** → improve the model

In [11]:
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_data,
    eval_dataset=test_data,
    data_collator=data_collator,
)

## 6. Start Training

In [ ]:
print("Starting training...")
trainer.train()
print("Training complete!")
print("Model saved to outputs/model/")

Starting training...


D:\4thGrade_01\Term02\ANLP\ResearchPaperSummarization(PDFs)_ANLP_Project\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
